# 04 · Visualization, differential expression and downstream analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/en/04_visualization_deg_downstream.ipynb)

**From Cells to Clusters: An Introduction to Single-Cell RNA-seq Analysis in Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Peru · November 9, 2026  
Instructor: Dr. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Time in the tutorial:** 11:35–12:05 (30 min)

**By the end of this notebook you will be able to:**
- Make publication-quality UMAPs, violin plots, dot plots, matrix plots and heatmaps
- Plot cell type proportions
- Find differentially expressed genes (DEGs) between cell types and draw a volcano plot
- Run over-representation analysis (MSigDB Hallmark) and pathway activity inference (PROGENy)
- Know why cell-level tests are **not** appropriate for comparing conditions

> 🟢 **Core** sections are covered live. 🔎 **Extra** sections are for self-study after the tutorial.

## 0. Setup 🟢
Run the cell below first (▶ or `Shift + Enter`). On Google Colab it installs the packages (~1 min);
on your own computer it assumes you already ran `pip install -r requirements.txt`.

**Optional:** set `USE_DRIVE = True` to save your results in Google Drive, so the next notebooks
can reuse them. If you leave it `False`, each notebook rebuilds what it needs automatically.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10" "decoupler>=2.1,<3"

USE_DRIVE = False  # True = keep results in Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Tutorial helper functions (download + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # running from a local clone
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import decoupler as dc
import seaborn as sns
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

In [ ]:
adata = tut.get_checkpoint("annotated", WORKDIR)
order = ["CD4 T", "CD8 T", "NK", "B", "CD14 Mono", "FCGR3A Mono", "DC", "Platelet"]
adata.obs["cell_type"] = adata.obs["cell_type"].cat.reorder_categories(
    [c for c in order if c in adata.obs["cell_type"].cat.categories])
markers = {k: v for k, v in tut.PBMC_MARKERS.items() if k in adata.obs["cell_type"].cat.categories}
adata

## 1. UMAP, the many ways 🟢
Tips: put labels on the data instead of a legend, use a perceptually uniform colormap (`viridis`, `magma`) for
expression, and order the points so that expressing cells are drawn on top (`sort_order=True`, the default).

In [ ]:
sc.pl.umap(adata, color="cell_type", legend_loc="on data", legend_fontsize=9,
           legend_fontoutline=2, title="PBMC 3k cell types")

In [ ]:
sc.pl.umap(adata, color=["MS4A1", "CD14", "GNLY", "FCGR3A"], cmap="magma", vmax="p99",
           ncols=4, use_raw=True)

In [ ]:
# Highlight one population
sc.pl.umap(adata, color="cell_type", groups=["CD14 Mono", "FCGR3A Mono"], title="Monocytes")
# Where do cells concentrate?
sc.tl.embedding_density(adata, basis="umap", groupby="cell_type")
sc.pl.embedding_density(adata, basis="umap", key="umap_density_cell_type",
                        group=["CD4 T", "B", "CD14 Mono"], ncols=3)

## 2. Expression across groups 🟢
| Plot | Best for |
|---|---|
| **Violin / stacked violin** | Full distribution of a few genes |
| **Dot plot** | Many genes × groups: % expressing + mean |
| **Matrix plot** | Mean expression only, compact |
| **Heatmap / tracks plot** | Individual cells, shows heterogeneity within a group |

In [ ]:
sc.pl.violin(adata, ["LYZ", "CD79A", "NKG7"], groupby="cell_type", rotation=60, use_raw=True)

In [ ]:
sc.pl.stacked_violin(adata, markers, groupby="cell_type", swap_axes=False, dendrogram=False, use_raw=True)

In [ ]:
sc.pl.dotplot(adata, markers, groupby="cell_type", standard_scale="var", use_raw=True,
              save="_pbmc_markers.pdf")   # saved in WORKDIR/figures/

In [ ]:
sc.pl.matrixplot(adata, markers, groupby="cell_type", standard_scale="var", cmap="Blues", use_raw=True)

In [ ]:
sc.pl.heatmap(adata, markers, groupby="cell_type", standard_scale="var", cmap="viridis",
              use_raw=True, swap_axes=True, figsize=(10, 6))

In [ ]:
sc.pl.tracksplot(adata, markers, groupby="cell_type", use_raw=True)

## 3. Cell type proportions 🟢
With one sample we can only describe composition. To **compare** proportions between conditions you need several
biological replicates per condition (notebook 05).

In [ ]:
props = adata.obs["cell_type"].value_counts(normalize=True).reindex(adata.obs["cell_type"].cat.categories)
colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"])) \
    if "cell_type_colors" in adata.uns else None
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={"width_ratios": [3, 1]})
props.mul(100).plot.bar(ax=axes[0], color=[colors[c] for c in props.index] if colors else None)
axes[0].set(ylabel="% of cells", xlabel="")
bottom = 0
for ct, p in props.items():
    axes[1].bar(0, p * 100, bottom=bottom, color=colors[ct] if colors else None, label=ct)
    bottom += p * 100
axes[1].set(xticks=[], ylabel="% of cells", title="PBMC 3k")
axes[1].legend(bbox_to_anchor=(1, 1), frameon=False, fontsize=8)
plt.tight_layout()

## 4. Differentially expressed genes between cell types 🟢
We test each cell type against all other cells. `pts=True` also reports the fraction of cells expressing each gene,
which helps you pick **specific** markers (high in the group, low elsewhere).

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="cell_type", method="wilcoxon", pts=True)
sc.pl.rank_genes_groups(adata, n_genes=15, ncols=4, sharey=False, fontsize=8)

In [ ]:
deg = sc.get.rank_genes_groups_df(adata, group="CD14 Mono")
deg_sig = deg.query("pvals_adj < 0.05 and abs(logfoldchanges) > 1")
print(f"{len(deg_sig)} significant genes ({(deg_sig.logfoldchanges > 0).sum()} up, {(deg_sig.logfoldchanges < 0).sum()} down)")
deg_sig.head(10)

In [ ]:
def volcano(df, title, lfc=1, padj=0.05, n_label=10):
    d = df.copy()
    d["nlp"] = -np.log10(d["pvals_adj"].clip(lower=1e-300))
    sig = (d["pvals_adj"] < padj) & (d["logfoldchanges"].abs() > lfc)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(d.loc[~sig, "logfoldchanges"], d.loc[~sig, "nlp"], s=4, c="lightgray")
    ax.scatter(d.loc[sig, "logfoldchanges"], d.loc[sig, "nlp"], s=6,
               c=np.where(d.loc[sig, "logfoldchanges"] > 0, "#d62728", "#1f77b4"))
    for _, r in d[sig].sort_values("nlp", ascending=False).head(n_label).iterrows():
        ax.annotate(r["names"], (r["logfoldchanges"], r["nlp"]), fontsize=8)
    ax.axvline(lfc, ls="--", c="k", lw=0.5); ax.axvline(-lfc, ls="--", c="k", lw=0.5)
    ax.axhline(-np.log10(padj), ls="--", c="k", lw=0.5)
    ax.set(xlabel="log2 fold change", ylabel="-log10 adjusted p-value", title=title)

volcano(deg, "CD14 monocytes vs. all other cells")

In [ ]:
# Keep only specific markers: expressed in >50% of the group and <20% of the rest
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, max_out_group_fraction=0.2,
                               min_fold_change=1, key_added="rank_genes_filtered")
sc.pl.rank_genes_groups_dotplot(adata, key="rank_genes_filtered", n_genes=4, standard_scale="var")

⚠️ **Cluster markers ≠ differential expression between conditions.** The Wilcoxon test treats every cell as an
independent replicate. Cells from the same person are not independent, so p-values become tiny and false positives
explode when you compare, e.g., patients vs. controls. For conditions, sum the counts per sample and cell type
(**pseudobulk**) and use bulk RNA-seq tools (DESeq2, edgeR). We do exactly that in notebook 05.

## 5. Downstream: what do the DEGs mean? 🟢
### 5.1 Over-representation analysis (ORA) with MSigDB Hallmark
Are the top markers of each cell type enriched in known biological processes? ORA is a Fisher's exact test between
your gene list and each gene set. We use the 50 **Hallmark** gene sets from MSigDB (via OmniPath).

In [ ]:
hallmark = tut.cached_table("hallmark_human", lambda: dc.op.hallmark(organism="human"), WORKDIR)
print(hallmark["source"].nunique(), "gene sets")

all_deg = sc.get.rank_genes_groups_df(adata, group=None)
# cell types x genes matrix of positive log fold changes
lfc = (all_deg.query("pvals_adj < 0.05").pivot_table(index="group", columns="names",
       values="logfoldchanges", observed=True).fillna(0).clip(lower=0))
ora_es, ora_p = dc.mt.ora(lfc, net=hallmark, n_up=100, tmin=5)
top_sets = (-np.log10(ora_p.clip(lower=1e-30))).max().sort_values(ascending=False).head(12).index
fig, ax = plt.subplots(figsize=(7, 0.35 * len(top_sets) + 1.5))
sns.heatmap((-np.log10(ora_p[top_sets].clip(lower=1e-30))).T.rename(index=lambda x: x.replace("HALLMARK_", "")),
            cmap="Reds", ax=ax, cbar_kws={"label": "-log10 adj. p"})
ax.set(xlabel="", ylabel="", title="Hallmark ORA on cell type markers")
plt.show()

### 5.2 Pathway activity per cell with PROGENy
**PROGENy** provides "footprints": genes that change when a signaling pathway (JAK-STAT, NF-κB, TNF-α, TGF-β...) is
activated. With a ULM we estimate an activity score for every cell. Unlike ORA, it uses the **direction and weight**
of each target gene.

In [ ]:
progeny = tut.cached_table("progeny_human", lambda: dc.op.progeny(organism="human", top=500), WORKDIR)
dc.mt.ulm(adata, net=progeny, raw=True)
pathway_acts = dc.pp.get_obsm(adata, "score_ulm")
sc.pl.matrixplot(pathway_acts, var_names=pathway_acts.var_names, groupby="cell_type",
                 cmap="RdBu_r", vcenter=0, colorbar_title="mean activity")

In [ ]:
sc.pl.umap(pathway_acts, color=[p for p in ["JAK-STAT", "NFkB", "TNFa"] if p in pathway_acts.var_names],
           cmap="RdBu_r", vcenter=0, ncols=3)

### 5.3 🔎 Extra: scoring your own gene signature
Any gene list (e.g. from a paper) can be scored with `sc.tl.score_genes`: the mean expression of the list minus the
mean of a random set of genes with similar expression. Here, an interferon-stimulated gene signature, which will be
much higher in the IFN-β-treated cells of notebook 05.

In [ ]:
isg = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "LY6E", "IFITM1", "IRF7", "STAT1"]
sc.tl.score_genes(adata, [g for g in isg if g in adata.raw.var_names], score_name="ISG_score", use_raw=True)
sc.pl.violin(adata, "ISG_score", groupby="cell_type", rotation=60, stripplot=False)

### 💡 Your turn
Repeat the DEG analysis comparing **CD14 Mono** directly against **FCGR3A Mono** (instead of against all cells)
and draw the volcano plot. *Hint:* `sc.tl.rank_genes_groups(..., groups=["CD14 Mono"], reference="FCGR3A Mono", key_added="mono")`
and `sc.get.rank_genes_groups_df(adata, group="CD14 Mono", key="mono")`.

In [ ]:
# Your code here

<details><summary><b>Show solution</b></summary>

```python
sc.tl.rank_genes_groups(adata, groupby="cell_type", groups=["CD14 Mono"], reference="FCGR3A Mono",
                        method="wilcoxon", key_added="mono")
volcano(sc.get.rank_genes_groups_df(adata, group="CD14 Mono", key="mono"), "CD14 vs FCGR3A monocytes")
```
</details>

## 6. Save

In [ ]:
for key in ["rank_genes_filtered", "mono"]:   # tables with empty values can't be saved to .h5ad
    adata.uns.pop(key, None)
adata.write_h5ad(f"{WORKDIR}/results/pbmc3k_final.h5ad")
print(os.listdir(f"{WORKDIR}/figures"))

---
## ✅ Done!
Last step: several samples, two conditions, and how Python compares with R. Next: **05 · Integration, conditions and R vs Python**.